In [ ]:
from transformers import PreTrainedTokenizerFast
import os

DATA_DIR = "/home/wyf/orcd/pool/reverse-llm/data"
TOKENIZER_DIR = "/home/wyf/orcd/pool/reverse-llm/tokenizers"

USER_ROLE_NAME = "user"[::-1]
ASSISTANT_ROLE_NAME = "assistant"[::-1]

dataset_name = "databricks-dolly"
context_length = 1024

In [3]:
tokenizer = PreTrainedTokenizerFast.from_pretrained(f"{TOKENIZER_DIR}/fineweb_bpe_200k")
tokenizer.add_special_tokens({ "additional_special_tokens": ["<im_start>", "<im_end>"] })

2

In [3]:
from datasets import Dataset, load_dataset
raw_dataset = load_dataset("databricks/databricks-dolly-15k", split="train")
split_datasets = raw_dataset.train_test_split(test_size=0.1, seed=0)

In [4]:
split_datasets

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 1502
    })
})

In [5]:
split_datasets["train"][0]

{'instruction': 'Why did Cato think that Carthage must be destroyed?',
 'context': 'Although Rome was successful in the first two Punic Wars, as it vied for dominance with the seafaring Punic city-state of Carthage in North Africa (now Tunisia), it suffered a number of humiliations and damaging reverses in the course of these engagements, especially at the Battle of Cannae in 216 BC. Rome nonetheless managed to win the Second Punic War thanks to Scipio Africanus in 201 BC. After its defeat, Carthage ceased to be a threat to Rome and was reduced to a small territory that was equivalent to what is now northeastern Tunisia.\n\nHowever, Cato the Censor visited Carthage in 152 BC as a member of a senatorial embassy, which was sent to arbitrate a conflict between the Punic city and Massinissa, the king of Numidia. Cato, a veteran of the Second Punic War, was shocked by Carthage\'s wealth, which he considered dangerous for Rome. He then relentlessly called for its destruction and ended all of

In [6]:
def process_chat_data(ds_split: Dataset):
    convos = []
    for ex in ds_split:
        instr = ex.get("instruction", "").strip()[::-1]
        ctx = ex.get("context", "").strip()[::-1]
        response = ex.get("response", "").strip()[::-1]

        user_msg_parts = [instr]
        if ctx != "":
            user_msg_parts.append(ctx)
        user_msg = "\n\n".join(user_msg_parts)

        if user_msg and response:
            convos.append([
                {"role": USER_ROLE_NAME, "content": user_msg},
                {"role": ASSISTANT_ROLE_NAME, "content": response},
            ])

    return convos

processed = {
    "train": process_chat_data(split_datasets["train"]),
    "valid": process_chat_data(split_datasets["test"]),
}

In [7]:
split_datasets["train"][0]

{'instruction': 'Why did Cato think that Carthage must be destroyed?',
 'context': 'Although Rome was successful in the first two Punic Wars, as it vied for dominance with the seafaring Punic city-state of Carthage in North Africa (now Tunisia), it suffered a number of humiliations and damaging reverses in the course of these engagements, especially at the Battle of Cannae in 216 BC. Rome nonetheless managed to win the Second Punic War thanks to Scipio Africanus in 201 BC. After its defeat, Carthage ceased to be a threat to Rome and was reduced to a small territory that was equivalent to what is now northeastern Tunisia.\n\nHowever, Cato the Censor visited Carthage in 152 BC as a member of a senatorial embassy, which was sent to arbitrate a conflict between the Punic city and Massinissa, the king of Numidia. Cato, a veteran of the Second Punic War, was shocked by Carthage\'s wealth, which he considered dangerous for Rome. He then relentlessly called for its destruction and ended all of

In [8]:
print(processed["train"][0])

[{'role': 'resu', 'content': '?deyortsed eb tsum egahtraC taht kniht otaC did yhW\n\n.)tse adnavres ogahtraC( "devas eb tsum egahtraC" ,esarhp emas eht htiw sehceeps sih lla dedne eh ,otaC ekiL .kcehc ni elpoep eht peek ot yrassecen saw ymene nommoc a fo raef eht taht deugra dna ytinu namoR evreserp ot raw eht desoppo mulucroC .rotanes laitneulfni tsom eht dna sunacirfA oipicS fo wal-ni-nos eht ,mulucroC acisaN oipicS suilenroC suilbuP yllaicepse ,hguoht mih wollof ot desufer etaneS ehT .rettam tnereffid yletelpmoc a no saw etabed eht nehw neve ,esarhp eht htiw sehceeps sih fo lla dedne dna noitcurtsed sti rof dellac ylsseltneler neht eH .emoR rof suoregnad deredisnoc eh hcihw ,htlaew s\'egahtraC yb dekcohs saw ,raW cinuP dnoceS eht fo naretev a ,otaC .aidimuN fo gnik eht ,assinissaM dna ytic cinuP eht neewteb tcilfnoc a etartibra ot tnes saw hcihw ,yssabme lairotanes a fo rebmem a sa CB 251 ni egahtraC detisiv rosneC eht otaC ,revewoH\n\n.aisinuT nretsaehtron won si tahw ot tnelaviuqe

In [9]:
tokenizer.chat_template = """{% for message in messages -%}
<im_start>{{ message['role'] }}
{{ message['content'] }}<im_end>
{%- endfor -%}
{% if add_generation_prompt and messages[-1]['role'] != 'assistant' -%}
<im_start>assistant
{%- endif %}"""

print(tokenizer.decode(tokenizer.apply_chat_template(processed["train"][2])))

<im_start>resu
?lacissalc naht tnereffid cisum zzaj si woH<im_end><im_start>tnatsissa
noitasivorpmi -
 ynomrah tnanossid fo ecnatpecca na -
sht31 dna ,sht11 ,sht7 ekil snoisnetxe elacs gnisu ynomrah xelpmoc -
 smhtyhr detapocnys -

 :era cisum zzaj fo serutaef tcnitsid emos ,revewoH  .zzaj enifed tonnac uoy taht dias evah snaicisum zzaj suomaf emoS .serutluc dna elpoep ssorca ediw era snoitinifed mrof tra yna ekil esuaceb ,enifed ot drah yllaer si sihT<im_end>


In [ ]:
from tqdm import tqdm

def filter_and_prepare_conversations(convos, tokenizer, max_len):
    filtered_convos = []
    for convo in tqdm(convos, desc="Filtering long conversations"):
        if not convo:
	        continue
        prompt_text = tokenizer.apply_chat_template(
            convo,
            tokenize=False,
            add_generation_prompt=False
        )
        tokenized_len = len(tokenizer.encode(prompt_text, truncation=False))

        if tokenized_len > 0 and tokenized_len <= max_len:
            filtered_convos.append(convo)
        elif tokenized_len == 0:
            print(f"Zero length: {convo}")
        
    return { "conversations": filtered_convos }


filtered_convos = {
    "train": filter_and_prepare_conversations(processed["train"], tokenizer, context_length),
    "valid": filter_and_prepare_conversations(processed["valid"], tokenizer, context_length),
}
print(len(filtered_convos["train"]["conversations"]))

Filtering long conversations: 100%|██████████| 1502/1502 [00:00<00:00, 3016.64it/s]

13346


In [18]:
def formatting_func(example):
    # 'example' here is something like {"conversations": [{"role": ..., "content": ...}, ...]}
    conversation = example["conversations"]

    prompt_text = tokenizer.apply_chat_template(
        conversation,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized_inputs = tokenizer(
        prompt_text,
        truncation=True, # not really needed here bcs we already remove those that > max length
        max_length=context_length,
        return_attention_mask=True,
        padding="max_length",
    )
    input_ids = tokenized_inputs["input_ids"]

    labels = [-100] * len(input_ids)
    
    # find the last assistant response
    last_assistant_idx = max(
        idx for idx, turn in enumerate(conversation)
        if turn["role"] == ASSISTANT_ROLE_NAME
    )

    # Use <im_start> and <im_end> tokens instead of BOS/EOS
    im_start_token_id = 52000  # <im_start>
    im_end_token_id = 52001    # <im_end>
    
    current_token_idx = 0
    for turn_idx, turn in enumerate(conversation):
        role = turn["role"]
        content = turn["content"]

        try:
            start_of_turn_bos_idx = input_ids.index(im_start_token_id, current_token_idx)
        except ValueError:
            break 

        search_for_eos_from = start_of_turn_bos_idx + 1 # search after the current <im_start>
        end_of_turn_eos_idx = -1

        for k_eos in range(search_for_eos_from, len(input_ids)):
            if input_ids[k_eos] == im_end_token_id:
                end_of_turn_eos_idx = k_eos
                break
        if end_of_turn_eos_idx == -1:
            print(f"Warning: Could not find <im_end> token for turn: {turn}")
            return None

        role_and_newline_text = f"{role}\n"
        role_and_newline_tokens = tokenizer.encode(role_and_newline_text, add_special_tokens=False)

        # The actual start of content tokens
        start_of_content_idx = start_of_turn_bos_idx + 1 + len(role_and_newline_tokens) # +1 for <im_start>

        if role == ASSISTANT_ROLE_NAME and turn_idx == last_assistant_idx:
            # unmask tokens from start_of_content_idx up to (but not including) end_of_turn_eos_idx
            for k_label in range(start_of_content_idx, end_of_turn_eos_idx + 1):
                if k_label >= 0 and k_label < len(labels):
                    labels[k_label] = input_ids[k_label]
        
        current_token_idx = end_of_turn_eos_idx + 1

    return {
        "input_ids": input_ids,
        "attention_mask": tokenized_inputs["attention_mask"],
        "labels": labels,
    }

print(dataset_name)

tokenized = {}
for split in ["train", "valid"]:
    tokenized[split] = (
        Dataset.from_dict(filtered_convos[split])
        .map(formatting_func)
        .select_columns(["input_ids", "attention_mask", "labels"])
    )
    tokenized[split].save_to_disk(f"{DATA_DIR}/{dataset_name}/tokenized_{context_length}_{split}")

databricks-dolly


Map:   0%|          | 0/13346 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/13346 [00:00<?, ? examples/s]

Map:   0%|          | 0/1472 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1472 [00:00<?, ? examples/s]

NameError: name 'model' is not defined